Bùi Nhật Huy 

In [1]:
import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit,StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from utils import *
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import wandb
import pandas as pd
from utils import *

from pathlib import Path

In [2]:
rf_model = joblib.load("../../../artifacts/Random_Forest.joblib")
rf_features = joblib.load("../../../artifacts/rf_features.joblib")

lgbm_model = joblib.load("../../../artifacts/lgbm_pipeline.joblib")   
lgbm_features = joblib.load("../../../artifacts/lgbm_features.joblib")

In [3]:

print("RF num features:", len(rf_features))
print("LGBM num features:", len(lgbm_features))
print("RF first 5:", rf_features)
print("LGBM first 5:", lgbm_features)

RF num features: 36
LGBM num features: 33
RF first 5: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', '_log_amount', 'is_night_proxy', 'is_business_hours_proxy', 'hour_sin', 'hour_cos', 'time_diff', 'is_high_amount', 'is_rapid_transaction']
LGBM first 5: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', '_log_amount', 'Hour_from_start_mod24', 'is_night_proxy', 'is_business_hours_proxy']


In [4]:
df = pd.read_csv('../../../data/creditcard.csv')

df = create_features(df)
df['_log_amount_raw'] = df['_log_amount']
scaler = StandardScaler()
df[['_log_amount']] = scaler.fit_transform(df[['_log_amount']])
df['hour_sin'] = np.sin(2 * np.pi * df['Hour_from_start_mod24']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['Hour_from_start_mod24']/24)
df['time_diff'] = df['Time'].diff().fillna(0)
threshold = df['Amount'].quantile(0.95)  
df['is_high_amount'] = (df['Amount'] > threshold).astype(int)
df['is_rapid_transaction'] = (df['time_diff'] < 60).astype(int)
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,_log_amount,Hour_from_start_mod24,is_night_proxy,is_business_hours_proxy,_log_amount_raw,hour_sin,hour_cos,time_diff,is_high_amount,is_rapid_transaction
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,1.123062,0,1,0,5.014760,0.0,1.0,0.0,0,1
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-1.115298,0,1,0,1.305626,0.0,1.0,0.0,0,1
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,1.680981,0,1,0,5.939276,0.0,1.0,1.0,1,1
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,1.008128,0,1,0,4.824306,0.0,1.0,0.0,0,1
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0.669117,0,1,0,4.262539,0.0,1.0,1.0,0,1


In [5]:
lgbm_features = ['_log_amount_raw' if c == '_log_amount' else c for c in lgbm_features]

In [6]:
all_features = sorted(set(rf_features) | set(lgbm_features))
print("Total union features:", len(all_features))

Total union features: 39


In [7]:
X_train, y_train, X_val, y_val, X_test, y_test = split_data(df, all_features, 'Class')

X_train: (181584, 39) y_train: (181584,)
X_val: (45396, 39) y_val: (45396,)
X_test: (56746, 39) y_test: (56746,)
Fraud rate in train: 0.001910961318177813
Fraud rate in test: 0.0013040566735981391


In [8]:
X_val_rf = X_val[rf_features].copy()
X_test_rf = X_test[rf_features].copy()

X_val_lgbm = X_val[lgbm_features].copy()
X_test_lgbm = X_test[lgbm_features].copy()

print("X_val_rf   :", X_val_rf.shape)
print("X_val_lgbm :", X_val_lgbm.shape)
print("X_test_rf  :", X_test_rf.shape)
print("X_test_lgbm:", X_test_lgbm.shape)

X_val_rf   : (45396, 36)
X_val_lgbm : (45396, 33)
X_test_rf  : (56746, 36)
X_test_lgbm: (56746, 33)


In [9]:
rf_val_proba = rf_model.predict_proba(X_val_rf)[:, 1]
rf_test_proba = rf_model.predict_proba(X_test_rf)[:, 1]

lgbm_val_proba = lgbm_model.predict_proba(X_val_lgbm)[:, 1]
lgbm_test_proba = lgbm_model.predict_proba(X_test_lgbm)[:, 1]

[LightGBM] [Warning] Unknown parameter: n_job
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] Unknown parameter: n_job
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0


### Emsenble 

In [12]:
ens_val_proba2080 = 0.2 * rf_val_proba + 0.8 * lgbm_val_proba
ens_val_proba5050 = 0.5 * rf_val_proba + 0.5 * lgbm_val_proba
ens_val_proba1090 = 0.1 * rf_val_proba + 0.9 * lgbm_val_proba
ens_val_proba3070 = 0.3 * rf_val_proba + 0.7 * lgbm_val_proba

In [13]:
rf_res = thr_for_precision(y_val, rf_val_proba)
lgbm_res = thr_for_precision(y_val, lgbm_val_proba)
ens_res2080 = thr_for_precision(y_val, ens_val_proba2080)
ens_res5050 = thr_for_precision(y_val, ens_val_proba5050)
ens_res1090 = thr_for_precision(y_val, ens_val_proba1090)
ens_res3070 = thr_for_precision(y_val, ens_val_proba3070)

rf_thr, rf_val_pr90, rf_val_recall = rf_res["threshold"], rf_res["precision"], rf_res["recall"]
lgbm_thr, lgbm_val_pr90, lgbm_val_recall = lgbm_res["threshold"], lgbm_res["precision"], lgbm_res["recall"]
ens_thr2080, ens_val_pr90_2080, ens_val_recall_2080 = ens_res2080["threshold"], ens_res2080["precision"], ens_res2080["recall"]
ens_thr5050, ens_val_pr90_5050, ens_val_recall_5050 = ens_res5050["threshold"], ens_res5050["precision"], ens_res5050["recall"]
ens_thr1090, ens_val_pr90_1090, ens_val_recall_1090 = ens_res1090["threshold"], ens_res1090["precision"], ens_res1090["recall"]
ens_thr3070, ens_val_pr90_3070, ens_val_recall_3070 = ens_res3070["threshold"], ens_res3070["precision"], ens_res3070["recall"]

thresholds_all = pd.DataFrame({
    'Model': ['Random Forest', 'LightGBM', 'Ensemble2080','Ensemble5050','Ensemble1090','Ensemble3070'],
    'Threshold': [rf_thr, lgbm_thr, ens_thr2080, ens_thr5050, ens_thr1090, ens_thr3070],
    'Precision@thr': [rf_val_pr90, lgbm_val_pr90, ens_val_pr90_2080,ens_val_pr90_5050,ens_val_pr90_1090,ens_val_pr90_3070],
    'Recall@thr': [rf_val_recall, lgbm_val_recall, ens_val_recall_2080,ens_val_recall_5050,ens_val_recall_1090,ens_val_recall_3070]
})

thresholds_all

,Model,Threshold,Precision@thr,Recall@thr
0,Random Forest,0.303112,0.906977,0.75
1,LightGBM,0.152935,0.906977,0.75
2,Ensemble2080,0.164633,0.906977,0.75
3,Ensemble5050,0.199137,0.906977,0.75
4,Ensemble1090,0.158784,0.906977,0.75
5,Ensemble3070,0.170482,0.906977,0.75


In [14]:
rf_val_metrics = evaluate(y_val, rf_val_proba, thr=rf_thr)
lgbm_val_metrics = evaluate(y_val, lgbm_val_proba, thr=lgbm_thr)
ens_val_metrics2080 = evaluate(y_val, ens_val_proba2080, thr=ens_thr2080)
ens_val_metrics5050 = evaluate(y_val, ens_val_proba5050, thr=ens_thr5050)
ens_val_metrics1090 = evaluate(y_val, ens_val_proba1090, thr=ens_thr1090)
ens_val_metrics3070 = evaluate(y_val, ens_val_proba3070, thr=ens_thr3070)
compare_df = pd.DataFrame([
    {"model": "RandomForest", **rf_val_metrics},
    {"model": "LightGBM", **lgbm_val_metrics},
    {"model": "Ensemble_20_80", **ens_val_metrics2080},
    {"model": "Ensemble_50_50", **ens_val_metrics5050},
    {"model": "Ensemble_10_90", **ens_val_metrics1090},
    {"model": "Ensemble_30_70", **ens_val_metrics3070}
])
compare_df


,model,threshold,precision,recall,f1,roc_auc,auprc,brier,tp,fp,fn,tn
0,RandomForest,0.303112,0.906977,0.75,0.821053,0.958305,0.751267,0.000391,39,4,13,45340
1,LightGBM,0.152935,0.906977,0.75,0.821053,0.980611,0.777368,0.000343,39,4,13,45340
2,Ensemble_20_80,0.164633,0.906977,0.75,0.821053,0.976168,0.764008,0.000341,39,4,13,45340
3,Ensemble_50_50,0.199137,0.906977,0.75,0.821053,0.976345,0.754603,0.000349,39,4,13,45340
4,Ensemble_10_90,0.158784,0.906977,0.75,0.821053,0.976133,0.765567,0.000341,39,4,13,45340
5,Ensemble_30_70,0.170482,0.906977,0.75,0.821053,0.976252,0.759732,0.000343,39,4,13,45340


In [15]:
rf_val_cost = realized_cost(y_val, rf_val_proba, thr=rf_thr)
lgbm_val_cost = realized_cost(y_val, lgbm_val_proba, thr=lgbm_thr)
ens_val_cost2080 = realized_cost(y_val, ens_val_proba2080, thr=ens_thr2080)
ens_val_cost5050 = realized_cost(y_val, ens_val_proba5050, thr=ens_thr5050)
ens_val_cost1090 = realized_cost(y_val, ens_val_proba1090, thr=ens_thr1090)
ens_val_cost3070 = realized_cost(y_val, ens_val_proba3070, thr=ens_thr3070)
cost_df = pd.DataFrame({
    'Model': ['Random Forest', 'LightGBM', 'Ensemble_20_80', 'Ensemble_50_50', 'Ensemble_10_90', 'Ensemble_30_70'],
    'Realized Cost': [rf_val_cost, lgbm_val_cost, ens_val_cost2080, ens_val_cost5050, ens_val_cost1090, ens_val_cost3070]
})
cost_df

,Model,Realized Cost
0,Random Forest,2620.0
1,LightGBM,2620.0
2,Ensemble_20_80,2620.0
3,Ensemble_50_50,2620.0
4,Ensemble_10_90,2620.0
5,Ensemble_30_70,2620.0


In [16]:
rf_val_eval = log_eval(y_val, rf_val_proba)
lgbm_val_eval = log_eval(y_val, lgbm_val_proba)
ens_val_eval2080 = log_eval(y_val, ens_val_proba2080)
ens_val_eval5050 = log_eval(y_val, ens_val_proba5050)
ens_val_eval1090 = log_eval(y_val, ens_val_proba1090)
ens_val_eval3070 = log_eval(y_val, ens_val_proba3070)
eval_df = pd.DataFrame([
    {"Model": "Random Forest", **rf_val_eval},
    {"Model": "LightGBM", **lgbm_val_eval},
    {"Model": "Ensemble_20_80", **ens_val_eval2080},
    {"Model": "Ensemble_50_50", **ens_val_eval5050},
    {"Model": "Ensemble_10_90", **ens_val_eval1090},
    {"Model": "Ensemble_30_70", **ens_val_eval3070}
])

eval_df

,Model,threshold,Cost,ROC_AUC,PR_AUC,debiased_ece,adaptive_ece,Brier
0,Random Forest,0.100,2340.0,0.958305,0.751267,0.001118,0.001047,0.000391
1,LightGBM,0.026,2465.0,0.980611,0.777368,0.000349,0.000224,0.000343
2,Ensemble_20_80,0.029,2325.0,0.976168,0.764008,0.000094,0.000105,0.000341
3,Ensemble_50_50,0.068,2315.0,0.976345,0.754603,0.000433,0.000401,0.000349
4,Ensemble_10_90,0.015,2345.0,0.976133,0.765567,0.000219,0.000110,0.000341
5,Ensemble_30_70,0.041,2320.0,0.976252,0.759732,0.000174,0.000204,0.000343


In [17]:
thr_df2080 = sweep_thresholds(y_val, ens_val_proba2080, cost=(COST_FP, COST_FN)).copy()
thr_df5050 = sweep_thresholds(y_val, ens_val_proba5050, cost=(COST_FP, COST_FN)).copy()
thr_df1090 = sweep_thresholds(y_val, ens_val_proba1090, cost=(COST_FP, COST_FN)).copy()
thr_df3070 = sweep_thresholds(y_val, ens_val_proba3070, cost=(COST_FP, COST_FN)).copy()
print("Số ngưỡng đã quét:", len(thr_df2080))
print("Số ngưỡng đã quét:", len(thr_df5050))
print("Số ngưỡng đã quét:", len(thr_df1090))
print("Số ngưỡng đã quét:", len(thr_df3070))

Số ngưỡng đã quét: 1001
Số ngưỡng đã quét: 1001
Số ngưỡng đã quét: 1001
Số ngưỡng đã quét: 1001


In [21]:
results = []

configs = {
    "20/80": (thr_df2080, ens_val_proba2080),
    "50/50": (thr_df5050, ens_val_proba5050),
    "10/90": (thr_df1090, ens_val_proba1090),
    "30/70": (thr_df3070, ens_val_proba3070),
}

for name, (df, proba) in configs.items():
    best_row = df.sort_values(
        by=["f1", "precision", "recall"],
        ascending=[False, False, False]
    ).iloc[0]

    best_thr = float(best_row["threshold"])

    metrics = evaluate(y_val, proba, thr=best_thr)

    results.append({
        "config": name,
        "threshold": best_thr,
        "f1": metrics["f1"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "roc_auc": metrics["roc_auc"],
        "auprc": metrics["auprc"],
        "brier": metrics["brier"],
        "tp": metrics["tp"],
        "fp": metrics["fp"],
        "fn": metrics["fn"],
        "tn": metrics["tn"],
    })

summary_df = pd.DataFrame(results)
summary_df = summary_df.sort_values(by=["f1", "precision"], ascending=False)
summary_df

,config,threshold,f1,precision,recall,roc_auc,auprc,brier,tp,fp,fn,tn
0,20/80,0.250,0.83871,0.95122,0.75,0.976168,0.764008,0.000341,39,2,13,45342
1,50/50,0.246,0.83871,0.95122,0.75,0.976345,0.754603,0.000349,39,2,13,45342
2,10/90,0.252,0.83871,0.95122,0.75,0.976133,0.765567,0.000341,39,2,13,45342
3,30/70,0.249,0.83871,0.95122,0.75,0.976252,0.759732,0.000343,39,2,13,45342


In [22]:
rows = []

for config_name, (thr_df, proba) in configs.items():
    candidate_df = thr_df[thr_df["precision"] > 0.90].copy()

    if len(candidate_df) == 0:
        rows.append({
            "config": config_name,
            "best_threshold": None,
            "precision": None,
            "recall": None,
            "f1": None,
            "roc_auc": None,
            "auprc": None,
            "brier": None,
            "tp": None,
            "fp": None,
            "fn": None,
            "tn": None,
            "note": "Không có threshold nào có precision > 0.90"
        })
        continue

    best_p90_row = candidate_df.sort_values(
        by=["f1", "recall", "threshold"],
        ascending=[False, False, True]
    ).iloc[0]

    best_p90_thr = float(best_p90_row["threshold"])
    metrics = evaluate(y_val, proba, thr=best_p90_thr)

    rows.append({
        "config": config_name,
        "best_threshold": best_p90_thr,
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "roc_auc": metrics["roc_auc"],
        "auprc": metrics["auprc"],
        "brier": metrics["brier"],
        "tp": metrics["tp"],
        "fp": metrics["fp"],
        "fn": metrics["fn"],
        "tn": metrics["tn"],
        "note": "OK"
    })

summary_p90_df = pd.DataFrame(rows)

summary_p90_df = summary_p90_df.sort_values(
    by=["f1", "recall"],
    ascending=[False, False],
    na_position="last"
).reset_index(drop=True)

print("\n=== Best threshold of each config with precision > 0.90 ===")
summary_p90_df


=== Best threshold of each config with precision > 0.90 ===


,config,best_threshold,precision,recall,f1,roc_auc,auprc,brier,tp,fp,fn,tn,note
0,20/80,0.250,0.95122,0.75,0.83871,0.976168,0.764008,0.000341,39,2,13,45342,OK
1,50/50,0.246,0.95122,0.75,0.83871,0.976345,0.754603,0.000349,39,2,13,45342,OK
2,10/90,0.252,0.95122,0.75,0.83871,0.976133,0.765567,0.000341,39,2,13,45342,OK
3,30/70,0.249,0.95122,0.75,0.83871,0.976252,0.759732,0.000343,39,2,13,45342,OK


In [24]:
rows2 = []
for name, (thr_df, proba) in configs.items():

    # ===== best F1 =====
    best_f1_row = thr_df.sort_values(
        by=["f1", "precision", "recall"],
        ascending=[False, False, False]
    ).iloc[0]

    best_f1_thr = float(best_f1_row["threshold"])

    rows2.append({
        "config": name,
        "rule": "best_f1",
        "threshold": best_f1_thr,
        "precision": best_f1_row["precision"],
        "recall": best_f1_row["recall"],
        "f1": best_f1_row["f1"],
        "cost": best_f1_row["cost"],
    })

    # ===== precision > 0.90 =====
    candidate_df = thr_df[thr_df["precision"] > 0.90].copy()

    if len(candidate_df) == 0:
        rows2.append({
            "config": name,
            "rule": "best_precision_gt_0.90",
            "threshold": None,
            "precision": None,
            "recall": None,
            "f1": None,
            "cost": None,
        })
    else:
        best_p90_row = candidate_df.sort_values(
            by=["f1", "recall", "threshold"],
            ascending=[False, False, True]
        ).iloc[0]

        rows2.append({
            "config": name,
            "rule": "best_precision_gt_0.90",
            "threshold": float(best_p90_row["threshold"]),
            "precision": best_p90_row["precision"],
            "recall": best_p90_row["recall"],
            "f1": best_p90_row["f1"],
            "cost": best_p90_row["cost"],
        })

summary_df = pd.DataFrame(rows2)

summary_df

,config,rule,threshold,precision,recall,f1,cost
0,20/80,best_f1,0.250,0.95122,0.75,0.83871,2610.0
1,20/80,best_precision_gt_0.90,0.250,0.95122,0.75,0.83871,2610.0
2,50/50,best_f1,0.246,0.95122,0.75,0.83871,2610.0
3,50/50,best_precision_gt_0.90,0.246,0.95122,0.75,0.83871,2610.0
4,10/90,best_f1,0.252,0.95122,0.75,0.83871,2610.0
5,10/90,best_precision_gt_0.90,0.252,0.95122,0.75,0.83871,2610.0
6,30/70,best_f1,0.249,0.95122,0.75,0.83871,2610.0
7,30/70,best_precision_gt_0.90,0.249,0.95122,0.75,0.83871,2610.0


<h4>best_thr = 0.25

In [25]:
ens_test_proba2080 = 0.2 * rf_test_proba + 0.8 * lgbm_test_proba

In [27]:
overall = evaluate(y_test, ens_test_proba2080, thr=0.25)
overall

{'threshold': 0.25,
 'precision': 0.8888888888888888,
 'recall': 0.7567567567567568,
 'f1': 0.8175182481751825,
 'roc_auc': 0.9858922657835701,
 'auprc': 0.8125893969359987,
 'brier': 0.0004096249555260354,
 'tp': 56,
 'fp': 7,
 'fn': 18,
 'tn': 56665}

In [28]:
overall = log_eval(y_test, ens_test_proba2080, thr=0.25)
overall

{'threshold': 0.25,
 'Cost': 3635.0,
 'Precision': 0.8888888888888888,
 'Recall': 0.7567567567567568,
 'F1': 0.8175182481751825,
 'ROC_AUC': 0.9858922657835701,
 'PR_AUC': 0.8125893969359987,
 'debiased_ece': 0.00014588342393387767,
 'adaptive_ece': 9.617671692498897e-05,
 'Brier': 0.0004096249555260354}